# Biological Data Drift Analysis & Monitoring Report
**Antibody HIC Risk Prediction Platform**
---

### 1. Introduction & Biological Context
In therapeutic antibody engineering, developability is as critical as target affinity. Hydrophobic Interaction Chromatography (HIC) retention time is a primary developability parameter. A retention time of **$\ge$ 11.5 minutes** indicates high hydrophobicity, which correlates strongly with aggregation, low solubility, and manufacturing failure.

To screen clones early, we trained a **ProtT5 + Logistic Regression (L2)** model on two critical physical descriptors:
1. **$T_{m,\text{app}}$ (Apparent Melting Temperature in °C):** Reflects thermal stability. Low thermal stability often points to structural instability and higher hydrophobic surface exposure.
2. **PSR Score (Polyspecificity Ready Score):** Measures non-specific binding. High PSR scores indicate high "stickiness".

### Why Data Drift Occurs in Production:
In high-throughput screening, a laboratory might shift its focus to:
* A different b-cell subset (e.g., naive vs. memory).
* Heavily engineered or mutated clone libraries (e.g., humanization or affinity maturation).
* A brand new biological target.

These shifts alter the underlying physical distributions of the antibodies being screened:
* **Thermal Stability Drift:** The average $T_{m,\text{app}}$ shifts downward.
* **PSR Drift:** The average PSR score shifts upward.

If our model is queried on these drifted populations, its predictions become unreliable. This notebook implements the **statistical smoke alarm** to detect this drift before predictions fail in production.

### 2. Methodology & Statistical Smoke Alarm (Kolmogorov-Smirnov Test)
To verify if incoming production queries align with our baseline training data, we employ the **Two-Sample Kolmogorov-Smirnov (KS) Test** as our statistical monitoring tool.

#### Conceptual Workflow:
Instead of inspecting individual sequences, the KS test compares the cumulative probability distributions (feature profiles) of two populations:
1. **The Baseline Reference:** The physical property distributions (thermal stability and non-specific binding) of our original 348 training antibodies.
2. **The Active Production Lot:** The physical property distributions of the new antibody candidates screened live via our API.

#### Operational Decision Rule:
The test calculates a **p-value** (ranging from 0.0 to 1.0) to measure the statistical similarity between the two populations. We apply a standard significance threshold of **0.05**:
* **p-value >= 0.05 (Target Met - Stable):** The production profiles are statistically consistent with the training baseline. The classification model operates within its trained boundaries and remains highly reliable.
* **p-value < 0.05 (Alert - Drifted):** The incoming physical distributions have drifted significantly. This activates our **MLOps Alert System**, indicating that prediction accuracy is compromised and model retraining is required.

#### Statistical Power Constraints (Sample Size):
To ensure mathematical validity, the system enforces a minimum sample size of **30 production queries** before running the comparison. Calculating distribution shifts on smaller samples yields low statistical power (high risk of false negatives), which could let severe biological drift go undetected.

In [8]:
import os
import sqlite3
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import plotly.express as px
from evidently import Report
from evidently.presets import DataDriftPreset

### 3. Load Datasets
We load:
1. **Reference Dataset:** The original processed training library of 348 baseline clones (`data/final_dataset.parquet`).
2. **Production Queries:** Loaded from our SQLite database (`src/production_logs.db`). If the database is empty or has fewer than 30 queries, we programmatically simulate 50 production queries (including drifted sequences) to satisfy statistical power and show how the detection behaves.

In [9]:
# Paths
ref_path = "../data/final_dataset.parquet"
db_path = "../src/production_logs.db"

# Fallback paths in case execution happens in project root
if not os.path.exists(ref_path):
    ref_path = "data/final_dataset.parquet"
    db_path = "src/production_logs.db"

# 1. Load Reference Dataset
ref_df = pd.read_parquet(ref_path)
print(f"Loaded Reference Dataset: {len(ref_df)} rows")

# Keep only physical parameters
ref_data = ref_df[["tm_app", "psr"]].copy()

# 2. Load or Simulate Production Logs
def get_production_data():
    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path)
        try:
            prod_df = pd.read_sql_query("SELECT tm_app, psr FROM api_logs WHERE status_code = 200", conn)
            conn.close()
            if len(prod_df) >= 30:
                print(f"Loaded {len(prod_df)} successful live logs from SQLite database.")
                return prod_df, False
        except Exception as e:
            print(f"Could not read logs from DB: {e}")
            conn.close()
            
    # Simulate production data if DB is empty or lacks logs
    print("Database empty or too small. Simulating 50 production screening requests...")
    print("- 30 stable baseline requests matching reference library")
    print("- 20 drifted requests representing a heavily engineered clone library (lower Tm, higher PSR)")
    
    np.random.seed(42)
    # 30 stable samples from reference
    stable_samples = ref_data.sample(30, random_state=42).copy()
    
    # 20 drifted samples: lower Tm, higher PSR
    drifted_samples = ref_data.sample(20, random_state=24).copy()
    drifted_samples["tm_app"] = drifted_samples["tm_app"] - 8.0  # Significant thermal stability loss
    drifted_samples["psr"] = drifted_samples["psr"] + 2.5       # Significant polyspecificity increase
    
    simulated_prod = pd.concat([stable_samples, drifted_samples], ignore_index=True)
    return simulated_prod, True

prod_data, is_simulated = get_production_data()

Loaded Reference Dataset: 348 rows
Loaded 46 successful live logs from SQLite database.


### 4. Running the Two-Sample Kolmogorov-Smirnov Test
Let's compute the p-value for `tm_app` and `psr` manually using `scipy.stats.ks_2samp` to understand the raw statistical scores.

In [14]:
alpha = 0.05

for col in ["tm_app", "psr"]:
    ref_values = ref_data[col].dropna()
    prod_values = prod_data[col].dropna()
    
    # Execute KS test
    ks_stat, p_value = ks_2samp(ref_values, prod_values)
    
    print(f"=== {col.upper()} Drift Test ===")
    print(f"KS Statistic: {ks_stat:.4f}")
    print(f"p-value:      {p_value:.10f}")
    
    if p_value < alpha:
        print(f"Verdict:      🔴 DRIFT DETECTED (p-value < {alpha})")
    else:
        print(f"Verdict:      🟢 STABLE (p-value >= {alpha})")
    print("-" * 40)

=== TM_APP Drift Test ===
KS Statistic: 0.4078
p-value:      0.0000015223
Verdict:      🔴 DRIFT DETECTED (p-value < 0.05)
----------------------------------------
=== PSR Drift Test ===
KS Statistic: 0.9397
p-value:      0.0000000000
Verdict:      🔴 DRIFT DETECTED (p-value < 0.05)
----------------------------------------


### 5. Generating the Evidently AI Data Drift Report
We will use **Evidently AI** to run a comprehensive, production-ready Data Drift Report. Evidently automatically handles the feature-level test selection and bundles them into a structured report.

In [11]:
# Build the Evidently Data Drift Report
report = Report(metrics=[DataDriftPreset()])
snapshot = report.run(reference_data=ref_data, current_data=prod_data)

# Extract structured results from the Snapshot
report_dict = snapshot.dict()
metrics_list = report_dict["metrics"]

# Extract dataset-level summary
count_metric = metrics_list[0]
drift_share = count_metric["value"]["share"]
drift_share_threshold = count_metric["config"]["drift_share"]
dataset_drifted = drift_share >= drift_share_threshold

print("=== Evidently AI Data Drift Verdict ===")
print(f"Drift Share:           {drift_share:.2%} (Threshold: {drift_share_threshold:.2%})")
if dataset_drifted:
    print("Dataset Status:        🔴 BIOLOGICAL DATA DRIFT DETECTED!")
else:
    print("Dataset Status:        🟢 BIOLOGICAL POPULATION STABLE")

=== Evidently AI Data Drift Verdict ===
Drift Share:           100.00% (Threshold: 50.00%)
Dataset Status:        🔴 BIOLOGICAL DATA DRIFT DETECTED!


### 6. Visualizing Distribution Comparisons
To see the physical overlap between our reference library and the production queries, we plot overlaying histograms with marginal box plots using **Plotly**. 

We strictly employ the **Okabe-Ito Color Palette** to guarantee full accessibility compliance under **WCAG 2.1 Success Criterion 1.4.1** (Use of Color):
* **Training Reference Library:** Deep Blue (`#0072B2`)
* **Live Production Queries:** Warm Orange (`#E69F00`)

In [12]:
# Align labels and prepare a single DataFrame for Plotly plotting
ref_plot = ref_data.copy()
ref_plot["Population"] = "Training Reference Library"

prod_plot = prod_data.copy()
prod_plot["Population"] = "Live Production Queries"

combined_df = pd.concat([ref_plot, prod_plot])

# Color mapping compliant with Okabe-Ito (Color-blind friendly)
color_map = {
    "Training Reference Library": "#0072B2", # Deep Blue
    "Live Production Queries": "#E69F00"     # Warm Orange
}

# 1. Plot apparent Melting Temperature (tm_app) Comparison
fig_tm = px.histogram(
    combined_df,
    x="tm_app",
    color="Population",
    barmode="overlay",
    nbins=30,
    marginal="box",
    title="Apparent Melting Temperature (tm_app) Distribution Comparison",
    labels={"tm_app": "apparent Melting Temperature (Tm,app in °C)"},
    color_discrete_map=color_map
)
fig_tm.update_layout(
    xaxis_title="apparent Melting Temperature (°C)",
    yaxis_title="Count",
    legend_title="Population Group"
)
fig_tm.show()

Let's also visualize the **PSR Score** distribution comparison.

In [13]:
# 2. Plot Polyspecificity Ready Score (psr) Comparison
fig_psr = px.histogram(
    combined_df,
    x="psr",
    color="Population",
    barmode="overlay",
    nbins=30,
    marginal="box",
    title="Polyspecificity Ready Score (psr) Distribution Comparison",
    labels={"psr": "Polyspecificity Ready Score (PSR)"},
    color_discrete_map=color_map
)
fig_psr.update_layout(
    xaxis_title="Polyspecificity Ready (PSR) Score",
    yaxis_title="Count",
    legend_title="Population Group"
)
fig_psr.show()

### 7. Interpretation & Biological Action Plan (Our SOP)

When the statistical smoke alarm sounds ($p\text{-value} < 0.05$), we execute our **MLOps Standard Operating Procedure (SOP)**:

#### 1. Biological Interpretation
Our ProtT5 classification model was trained on a reference library with specific structural rules. 
* A drop in average $T_{m,\text{app}}$ indicates that the laboratory has shifted to less thermally stable antibodies, which are prone to unfolding and exposing hydrophobic core segments.
* A spike in average $PSR$ means the screened candidates are significantly stickier (higher non-specific binding).
* Because these physical parameter ranges fall outside our model's training boundaries, **the model's predictions on these candidates are highly unreliable and likely to underestimate risk.**

#### 2. What action should be taken? 
1. **Alert the Lab & Product Owner:** Notify the team that the model is operating in a "drifted state" and predictions on the latest lot must be treated with caution.
2. **Collect & Annotate Logs:** Extract the accumulated query inputs from `production_logs.db`. Route these clones to wet-lab chromatography to obtain physical HIC retention times (ground-truth labels).
3. **Model Retraining:** Once ground-truth labels are acquired, retrain the **Logistic Regression** model using a combined dataset containing both the original 348 reference antibodies and the newly drifted clones. This expands the model's physical boundary knowledge.
4. **Automated Deployment:** Commit the newly trained model artifact (`best_model.joblib`) and push to main, triggering our GitHub Actions CI/CD to automatically test, build, and deploy the updated container.